# 추천 시스템 개발 

In [12]:
# 1. 현재 꼬여있는 NumPy와 관련 패키지 강제 삭제
!pip uninstall -y numpy scikit-surprise torch

# 2. 호환성이 검증된 NumPy 1.x 버전과 라이브러리 재설치
# 1.26.4는 NumPy 1세대의 마지막 안정화 버전입니다.
!pip install "numpy<2" scikit-surprise torch

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4
  Using cached scikit_surprise-1.1.4-cp312-cp312-macosx_10_9_universal2.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 36.8 MB/s  0:00:007.4 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 17.4 MB/s  0:00:08 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-surprise]1/3 [torch]


In [31]:
# 1. 대상 폴더가 없다면 생성합니다.
!mkdir -p /Users/gangseongbin/.cache/torch/hub/checkpoints/

# 2. 다운로드 폴더에 있는 파일을 해당 위치로 이동시킵니다.
!mv ~/Downloads/resnet101-cd907fc2.pth /Users/gangseongbin/.cache/torch/hub/checkpoints/

mv: rename /Users/gangseongbin/Downloads/resnet101-cd907fc2.pth to /Users/gangseongbin/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth: No such file or directory


## 라이브러리
| 분류 | 라이브러리 |
|-----|----------|
| 딥러닝 | torch, nn, optim |
| 이미지 처리 | torchvision, transforms |
| 데이터 로딩 | DataLoader, datasets |
| 수치 연산 | numpy |
| 시각화 | matplotlib |
| 파일 관리 | shutil, os |
| 진행률 표시 | tqdm |
| 디버깅 | set_trace |
| 수학 | math |
| 시간 측정 | time |

## 🧾 한 줄 정리
> 해당 라이브러리들은 이미지 분류 기반 딥러닝 프로젝트에서 데이터 처리, 모델 학습, 시각화, 디버깅까지 전체 파이프라인을 구성하는 핵심 도구들이다.

In [5]:
!pip install --upgrade certifi

In [3]:
import torch
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import os
import random
import ssl
import certifi
import os

print(os.listdir("./data"))

['food-101.tar.gz']


In [4]:
base_path = "/Users/gangseongbin/login-project/src/data/collection/data"

for root, dirs, files in os.walk(base_path):

    print(root)

/Users/gangseongbin/login-project/src/data/collection/data


## 2. MPS(GPU) 장치 설정

In [5]:
ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=certifi.where()
)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"

)

print("현재 사용 장치:", device)
# 모델을 GPU로 보내기
model = models.resnet101(weights=None)
model.to(device)

현재 사용 장치: mps


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

## 3. 데이터 전처리(Transform)

In [6]:
# 이미지를 Tensor 형태로 변환

transform = transforms.Compose([
    # CNN 입력 크기 통일
    transforms.Resize((224, 224)),
    # Tensor 변환
    transforms.ToTensor(),
    # 정규화
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

## 4. Food101 데이터셋 다운로드 및 로드

root
- 데이터가 저장될 폴더
- split='train'
- 학습용 데이터 사용

In [ ]:
food_dataset = datasets.Food101(
    root="/Users/gangseongbin/login-project/src/data/collection/data",
    split="train",
    download=True,
    transform=transform
)

  5%|█▊                                   | 239271936/4996278331 [13:29<5:28:07, 241622.38it/s]

## 4. Food101 클래스(카테고리) 확인
> Food101은 총 101개의 음식 카테고리를 보유

In [1]:
classes = food_dataset.classes
print("총 음식 클래스 개수:")
print(len(classes))
print("\n일부 음식 카테고리 확인:")
print(classes[:20])

NameError: name 'food_dataset' is not defined